# 07. Advanced Features

## Learning Objectives
- Implement a Human-in-the-Loop workflow
- Understand streaming modes and the namespace system
- Understand sandbox integrations such as Modal, Daytona, and Runloop
- Learn how ACP (Agent Client Protocol) connects agents to editors
- Learn how to use the Deep Agents CLI


In [ ]:
# Environment setup
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set!"
print("Environment setup complete")


In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

print(f"Model configured: {model.model_name}")


---
## 1. Human-in-the-Loop (HITL)

Human-in-the-Loop is a workflow in which the agent **requires human approval** before calling sensitive tools.

### How it Works

![Human-in-the-Loop workflow](../assets/images/hitl_flow.png)

### Required Condition
- **Checkpointer**: required to preserve the agent's state between interrupt and resume


In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver

# Choose which tools require approval with interrupt_on
hitl_agent = create_deep_agent(
    model=model,
    system_prompt="You are a file management assistant. Respond in English.",
    checkpointer=MemorySaver(),  # required!
    interrupt_on={
        "write_file": True,
        "edit_file": True,
    },
)

print("Human-in-the-Loop agent created")
print("write_file and edit_file now require approval")


In [ ]:
# Run the agent — it will pause before write_file
config = {"configurable": {"thread_id": "hitl-demo"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Create a file named config.yaml and write 'debug: true'."}]},
    config={**config, **lf_config},
)

# Inspect interrupts
if "__interrupt__" in result:
    interrupt_info = result["__interrupt__"]
    print("The agent was interrupted")
    print(f"Pending approvals: {len(interrupt_info)}")
    for item in interrupt_info:
        val = item.value if hasattr(item, 'value') else item
        print(f"  - Interrupt payload: {val}")
else:
    print("Completed without interruption:")
    print(result["messages"][-1].content)


In [ ]:
from langgraph.types import Command

# Resume with approval
if "__interrupt__" in result:
    resumed = hitl_agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                    # {"type": "reject"}
                    # {"type": "edit", "args": {"content": "debug: false"}}
                ]
            }
        ),
        config={**config, **lf_config},
    )

    print("✅ Resumed after approval:")
    print(resumed["messages"][-1].content)


---
## 2. Advanced Streaming

Deep Agents runs on top of LangGraph's streaming infrastructure.

### Stream Modes

| Mode | Description | Use Case |
|------|------|-------------|
| `"updates"` | State updates after each node finishes | Progress tracking |
| `"messages"` | Token-level streaming | Real-time text output |
| `"custom"` | Events emitted inside tools or nodes | Custom progress reporting |

### Namespace System

Events from subagents are separated by namespace:

```python
()                                # main agent
("tools:abc123",)                # subagent (tool call ID)
("tools:abc123", "model:def456")  # inner node inside a subagent
```


In [ ]:
from typing import Literal
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY", ""))


def internet_search(
    query: str,
    max_results: int = 3,
    topic: Literal["general", "news"] = "general",
) -> dict:
    """Search the internet for information."""
    return tavily_client.search(query, max_results=max_results, topic=topic)


stream_agent = create_deep_agent(
    model=model,
    system_prompt="You are a research coordinator. Respond in English.",
    subagents=[
        {
            "name": "researcher",
            "description": "Uses internet search to investigate topics.",
            "system_prompt": "Search the internet, collect the requested information, and summarize it concisely.",
            "tools": [internet_search],
        }
    ],
)

print("Streaming demo agent created")


In [ ]:
# Stream events with subgraphs — use namespaces to distinguish sources
print("=== Subagent event streaming ===")
print()

for namespace, chunk in stream_agent.stream(
    {"messages": [{"role": "user", "content": "Research the latest LangGraph features."}]},
    stream_mode="updates",
    subgraphs=True,
    config=lf_config,
):
    source = "[main]" if not namespace else f"[subagent: {namespace}]"

    for node_name, node_data in chunk.items():
        if not node_data:
            continue
        msgs = node_data.get("messages", [])
        if hasattr(msgs, "value"):
            msgs = msgs.value
        if msgs:
            last_msg = msgs[-1]
            if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    print(f"{source} 🔧 tool call: {tc['name']}")
            elif hasattr(last_msg, "content") and last_msg.content:
                content = last_msg.content if isinstance(last_msg.content, str) else str(last_msg.content)
                if content.strip() and not hasattr(last_msg, "tool_call_id"):
                    preview = content[:100].replace("\n", " ")
                    print(f"{source} 💬 {preview}...")


In [ ]:
# Use multiple stream modes together
print("=== Multiple stream modes ===")
print()

for event in stream_agent.stream(
    {"messages": [{"role": "user", "content": "Explain one new feature in Python 3.13 in a single sentence."}]},
    stream_mode=["updates", "messages"],
    subgraphs=True,
    config=lf_config,
):
    *namespace_parts, mode, data = event
    if mode == "updates":
        for node_name in data:
            print(f"  [updates] node '{node_name}' completed")
    elif mode == "messages":
        msg, metadata = data
        if hasattr(msg, "content") and msg.content and metadata and metadata.get("langgraph_node") == "model":
            print(msg.content, end="", flush=True)

print()


---
## 3. Sandboxes

A sandbox lets the agent run code in an **isolated environment**.  
That prevents it from accessing the host machine's files, network, or credentials directly.

### Supported Providers

| Provider | Characteristics | Best Fit |
|-----------|------|------------|
| **Modal** | GPU support, ML workloads | AI / ML tasks |
| **Daytona** | TypeScript / Python, fast cold starts | Web development |
| **Runloop** | Disposable devboxes, isolated execution | Code testing |

### Architecture Pattern

**Use the sandbox as a tool** (recommended)

![Sandbox architecture](../assets/images/sandbox_architecture.png)

### ⚠️ Security Guidelines
- **Never put secrets inside the sandbox** — the agent may leak them
- Manage credentials only through external tools
- Use Human-in-the-Loop approval for sensitive operations
- Block unnecessary network access


In [ ]:
# Sandbox integration example (reference only — requires provider-specific setup)

sandbox_example_code = """
# pip install deepagents-modal
from deepagents.backends.sandbox import ModalSandbox

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    backend=ModalSandbox(
        image="python:3.12-slim",
        gpu="T4",  # GPU support
    ),
)
"""

print("Sandbox integration example (reference only):")
print(sandbox_example_code)


---
## 4. ACP (Agent Client Protocol)

ACP standardizes communication **between coding agents and editors / IDEs**.

### Supported Editors
- **Zed** — native integration
- **JetBrains IDEs** — built-in support
- **VS Code** — `vscode-acp` plugin
- **Neovim** — ACP-compatible plugins

### MCP vs ACP
| Protocol | Purpose |
|---------|------|
| MCP (Model Context Protocol) | External tool integration |
| ACP (Agent Client Protocol) | Editor ↔ agent integration |


In [ ]:
# ACP server implementation example (reference only)
acp_example_code = """
# pip install deepagents-acp
from deepagents import create_deep_agent
from deepagents_acp import AgentServerACP
from langgraph.checkpoint.memory import MemorySaver

# Create the agent
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    system_prompt="You are a coding assistant.",
    checkpointer=MemorySaver(),
)

# Run the ACP server (stdio mode)
server = AgentServerACP(agent)
server.run()
"""

print("ACP server implementation example (reference only):")
print(acp_example_code)


---
## 5. Deep Agents CLI

The Deep Agents CLI is a **terminal coding agent** built on top of the SDK.

### Installation and Execution
```bash
# Install
uv tool install deepagents-cli

# Run
deepagents-cli

# Run directly without installing
uvx deepagents-cli
```

### Main Options

| Option | Description |
|------|------|
| `-a/--agent AGENT` | Specify the agent name |
| `-M/--model MODEL` | Choose the model |
| `-n/--non-interactive` | Non-interactive mode (single task execution) |
| `--auto-approve` | Skip human confirmation |
| `--sandbox {none,modal,daytona,runloop}` | Select a sandbox backend |

### Interactive Commands

| Command | Description |
|------|------|
| `/model` | Change the model |
| `/remember` | Store information in memory |
| `/tokens` | Inspect token usage |
| `!command` | Run a shell command |

### Memory System
- **Global**: `~/.deepagents/<agent_name>/memories/`
- **Project**: `.deepagents/AGENTS.md` (project root)


In [ ]:
# CLI non-interactive examples (run in the shell)
cli_examples = """
# Basic usage
deepagents-cli

# Non-interactive execution with a specific model
deepagents-cli -M claude-sonnet-4-6 -n "Write the README.md file for this project"

# Run inside a sandbox
deepagents-cli --sandbox modal "Run the test suite"

# Skill management
deepagents-cli skills list
deepagents-cli skills create my-skill
"""

print("CLI usage examples (run in the terminal):")
print(cli_examples)


---
## Full Track Summary

| Notebook | Topic | Key APIs |
|--------|------|----------|
| **01** | Introduction | `deepagents.__version__` |
| **02** | Quickstart | `create_deep_agent()`, `invoke()`, `stream()` |
| **03** | Customization | `model`, `system_prompt`, `tools`, `response_format` |
| **04** | Backends | `StateBackend`, `FilesystemBackend`, `StoreBackend`, `CompositeBackend` |
| **05** | Subagents | `SubAgent`, `CompiledSubAgent`, `subagents` |
| **06** | Memory & Skills | `memory`, `skills`, `AGENTS.md`, `SKILL.md` |
| **07** | Advanced Features | `interrupt_on`, `stream_mode`, Sandbox, ACP, CLI |

### Next Steps
→ Continue to **[08_harness.ipynb](./08_harness.ipynb)**
→ Or jump to the **advanced track** at **[../05_advanced/00_migration.ipynb](../05_advanced/00_migration.ipynb)**
